# Section 1: Configuration & Authentication

First, let's set up our configuration variables and verify authentication.

In [30]:
# Configuration variables

PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           


print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")

📋 Configuration:
  Project ID: johnswain-1200-20260303091357


In [31]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

🔐 Authentication Status:
  ✅ Authenticated successfully
  ✅ Default project: graph-demo-471710
  ⚠️  Note: Using PROJECT_ID (johnswain-1200-20260303091357) instead of default (graph-demo-471710)


# Section 2: Create BigQuery Dataset and Tables


## Copy Census Bureau ACS Dataset

This section copies all tables from the public BigQuery dataset `bigquery-public-data.census_bureau_acs` into your project.

**What's included:**
- ~280 tables containing US Census Bureau American Community Survey data
- Tables include blockgroup, cbsa (Core Based Statistical Area), and census tract data
- Data spans multiple years (2007-2018) and survey periods (1-year, 3-year, 5-year)

**Process:**
1. Install required libraries
2. Create destination dataset in your project
3. List all tables from source dataset
4. Copy tables individually (more reliable than Data Transfer Service)
5. Validate completion

**Estimated time:** 10-30 minutes depending on number of tables

**Note:** We use direct table copying rather than Data Transfer Service for simpler setup without requiring service account configuration.

In [32]:
# Install required libraries for BigQuery Data Transfer
import sys
import subprocess

# Install the packages
packages = ["google-cloud-bigquery-datatransfer", "google-cloud-bigquery"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")
print()
print("⚠️  IMPORTANT: After running this cell, please restart the kernel:")
print("   - Click 'Kernel' → 'Restart' in the menu")
print("   - Or use the restart button in the toolbar")
print("   - Then continue running from the next cell")
print()
print("   This ensures the newly installed packages are available.")


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


✅ Libraries installed

⚠️  IMPORTANT: After running this cell, please restart the kernel:
   - Click 'Kernel' → 'Restart' in the menu
   - Or use the restart button in the toolbar
   - Then continue running from the next cell

   This ensures the newly installed packages are available.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [33]:
# Alternative: Reload packages without kernel restart (try this first)
import sys
import importlib

# Force reload the package path
if 'google.cloud.bigquery_datatransfer' in sys.modules:
    del sys.modules['google.cloud.bigquery_datatransfer']
if 'google.cloud.bigquery' in sys.modules:
    del sys.modules['google.cloud.bigquery']

# Now import
try:
    from google.cloud import bigquery
    from google.cloud import bigquery_datatransfer
    
    # Initialize BigQuery client
    bq_client = bigquery.Client(project=PROJECT_ID)
    
    # Dataset configuration
    DATASET_ID = "census_bureau_acs"
    DATASET_LOCATION = "US"  # Must match source dataset location
    
    # Create dataset
    dataset_id = f"{PROJECT_ID}.{DATASET_ID}"
    dataset = bigquery.Dataset(dataset_id)
    dataset.location = DATASET_LOCATION
    
    dataset = bq_client.create_dataset(dataset, exists_ok=True)
    print(f"✅ Dataset created/verified: {dataset_id}")
    print(f"   Location: {DATASET_LOCATION}")
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print()
    print("Please restart the kernel and try again:")
    print("   - Click 'Kernel' → 'Restart' in the menu")
    print("   - Then run cells 1-3 again (config and auth)")
    print("   - Skip cell 6 (already installed)")
    print("   - Continue from this cell")
    raise

✅ Dataset created/verified: johnswain-1200-20260303091357.census_bureau_acs
   Location: US


In [34]:
# List all tables in source dataset to copy
# Using direct table copying instead of Data Transfer Service (simpler setup)

SOURCE_PROJECT_ID = "bigquery-public-data"
SOURCE_DATASET_ID = "census_bureau_acs"
source_dataset_ref = f"{SOURCE_PROJECT_ID}.{SOURCE_DATASET_ID}"

print(f"📋 Preparing to copy tables:")
print(f"   Source: {source_dataset_ref}")
print(f"   Destination: {dataset_id}")
print()
print("🔍 Listing tables in source dataset...")

# List all tables in the source dataset
source_tables = list(bq_client.list_tables(source_dataset_ref))
total_tables = len(source_tables)

print(f"   Found {total_tables} tables in source")
print()

# Check which tables already exist in destination
print("🔍 Checking for existing tables in destination...")
existing_tables = list(bq_client.list_tables(dataset_id))
existing_table_ids = {table.table_id for table in existing_tables}
existing_count = len(existing_table_ids)

print(f"   Found {existing_count} tables already in destination")
print()

# Filter to only tables that don't exist in destination
tables_to_copy = [table for table in source_tables 
                  if table.table_id not in existing_table_ids]
tables_to_copy_count = len(tables_to_copy)

# Report status
if existing_count > 0:
    print(f"📊 Copy Status:")
    print(f"   ✅ Already copied: {existing_count} tables")
    print(f"   📋 Remaining to copy: {tables_to_copy_count} tables")
    print()

if tables_to_copy_count == 0:
    print("✅ All tables already exist in destination - nothing to copy!")
else:
    # Show first 10 tables to be copied as preview
    print(f"📋 First 10 tables to copy:")
    for i, table in enumerate(tables_to_copy[:10]):
        print(f"   {i+1}. {table.table_id}")
    if tables_to_copy_count > 10:
        print(f"   ... and {tables_to_copy_count - 10} more")
    
    print()
    print(f"✅ Ready to copy {tables_to_copy_count} tables")

📋 Preparing to copy tables:
   Source: bigquery-public-data.census_bureau_acs
   Destination: johnswain-1200-20260303091357.census_bureau_acs

🔍 Listing tables in source dataset...
   Found 278 tables in source

🔍 Checking for existing tables in destination...
   Found 278 tables already in destination

📊 Copy Status:
   ✅ Already copied: 278 tables
   📋 Remaining to copy: 0 tables

✅ All tables already exist in destination - nothing to copy!


In [35]:
# Copy all tables from source to destination
import time

# Check if there are any tables to copy
if tables_to_copy_count == 0:
    print("✅ No tables to copy - all tables already exist in destination!")
else:
    print("🚀 Starting table copy operations...")
    print(f"   Copying {tables_to_copy_count} tables (skipping {existing_count} already copied)")
    print()

    # Track progress
    copied_tables = []
    failed_tables = []

    # Copy tables in batches
    for i, source_table in enumerate(tables_to_copy):
        table_name = source_table.table_id
        source_table_ref = f"{SOURCE_PROJECT_ID}.{SOURCE_DATASET_ID}.{table_name}"
        dest_table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
        
        try:
            # Create copy job
            job_config = bigquery.CopyJobConfig()
            job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
            
            copy_job = bq_client.copy_table(
                source_table_ref,
                dest_table_ref,
                job_config=job_config
            )
            
            # Wait for job to complete (with timeout)
            copy_job.result(timeout=300)  # 5 minute timeout per table
            
            copied_tables.append(table_name)
            
            # Print progress every 10 tables
            if (i + 1) % 10 == 0:
                print(f"   ✅ Copied {i + 1}/{tables_to_copy_count} tables...")
            
        except Exception as e:
            print(f"   ❌ Failed to copy {table_name}: {str(e)[:100]}")
            failed_tables.append((table_name, str(e)))
            continue

    print()
    print("=" * 60)
    print(f"✅ Copy operation completed!")
    print(f"   Successfully copied: {len(copied_tables)} tables")
    if failed_tables:
        print(f"   Failed: {len(failed_tables)} tables")
    print(f"   Total in destination: {existing_count + len(copied_tables)} tables")
    print("=" * 60)

✅ No tables to copy - all tables already exist in destination!


In [36]:
# Show any failed tables
if failed_tables:
    print()
    print("⚠️  Failed Tables:")
    for table_name, error in failed_tables[:10]:  # Show first 10
        print(f"   - {table_name}: {error[:80]}...")
    if len(failed_tables) > 10:
        print(f"   ... and {len(failed_tables) - 10} more failures")
else:
    print()
    print("✅ All tables copied successfully!")


✅ All tables copied successfully!


In [37]:
# Validate dataset copy
print("🔍 Validating copied dataset...")
print()

# List tables in destination dataset
dest_tables = list(bq_client.list_tables(dataset_id))
table_count = len(dest_tables)

print(f"✅ Destination dataset: {dataset_id}")
print(f"   Tables in destination: {table_count}")
print(f"   Expected: {total_tables}")
print()

# Show first 10 tables as sample
print("📋 Sample tables in destination:")
for i, table in enumerate(dest_tables[:10]):
    print(f"   {i+1}. {table.table_id}")

if table_count > 10:
    print(f"   ... and {table_count - 10} more tables")

print()

# Verify we have the expected number of tables
if table_count >= total_tables - 5:  # Allow small variance
    print(f"✅ Checkpoint passed: Dataset copied successfully!")
    print(f"   Expected {total_tables} tables, found {table_count}")
else:
    print(f"⚠️  Warning: Expected {total_tables} tables but found {table_count}")
    print(f"   Some tables may not have been copied")

print()
print(f"🔗 View in Console:")
print(f"   https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{DATASET_ID}!3sblockgroup_2018_5yr")

🔍 Validating copied dataset...

✅ Destination dataset: johnswain-1200-20260303091357.census_bureau_acs
   Tables in destination: 278
   Expected: 278

📋 Sample tables in destination:
   1. blockgroup_2010_5yr
   2. blockgroup_2011_5yr
   3. blockgroup_2012_5yr
   4. blockgroup_2013_5yr
   5. blockgroup_2014_5yr
   6. blockgroup_2015_5yr
   7. blockgroup_2016_5yr
   8. blockgroup_2017_5yr
   9. blockgroup_2018_5yr
   10. cbsa_2007_1yr
   ... and 268 more tables

✅ Checkpoint passed: Dataset copied successfully!
   Expected 278 tables, found 278

🔗 View in Console:
   https://console.cloud.google.com/bigquery?project=johnswain-1200-20260303091357&ws=!1m5!1m4!4m3!1sjohnswain-1200-20260303091357!2scensus_bureau_acs!3sblockgroup_2018_5yr


In [38]:
# View dataset in BigQuery Console
print()
print("🔗 Quick Links:")
print(f"   Dataset: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&d={DATASET_ID}&p={PROJECT_ID}&page=dataset")
print(f"   Sample table: https://console.cloud.google.com/bigquery?project={PROJECT_ID}&ws=!1m5!1m4!4m3!1s{PROJECT_ID}!2s{DATASET_ID}!3sblockgroup_2018_5yr")
print()
print("✅ Section 2 complete! Census Bureau ACS dataset is now available in your project.")


🔗 Quick Links:
   Dataset: https://console.cloud.google.com/bigquery?project=johnswain-1200-20260303091357&d=census_bureau_acs&p=johnswain-1200-20260303091357&page=dataset
   Sample table: https://console.cloud.google.com/bigquery?project=johnswain-1200-20260303091357&ws=!1m5!1m4!4m3!1sjohnswain-1200-20260303091357!2scensus_bureau_acs!3sblockgroup_2018_5yr

✅ Section 2 complete! Census Bureau ACS dataset is now available in your project.


# Section 3: Create Dataplex Aspect Type

This section creates a custom Dataplex Aspect Type to capture census-specific metadata for the ACS dataset tables.

## Create Census Survey Metadata Aspect Type

An **Aspect Type** in Dataplex defines a reusable metadata template that can be attached to data assets. This aspect type will capture:

- **Survey Vintage**: The year the census data represents (e.g., 2022)
- **Estimate Period**: Whether it's a 1-year or 5-year estimate
- **Experimental Data**: Flag to indicate if the Census Bureau labeled this as experimental

This metadata enriches the data catalog and makes it easier to understand the nature and quality of each dataset.

In [39]:
# Install Dataplex library
import sys
import subprocess

print("📦 Installing Dataplex library...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "google-cloud-dataplex"])

print("✅ Library installed")
print()
print("⚠️  IMPORTANT: After running this cell, please restart the kernel:")
print("   - Click 'Kernel' → 'Restart' in the menu")
print("   - Or use the restart button in the toolbar")
print("   - Then continue running from the next cell")
print()
print("   This ensures the newly installed package is available.")

📦 Installing Dataplex library...
✅ Library installed

⚠️  IMPORTANT: After running this cell, please restart the kernel:
   - Click 'Kernel' → 'Restart' in the menu
   - Or use the restart button in the toolbar
   - Then continue running from the next cell

   This ensures the newly installed package is available.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [40]:
# Alternative: Reload packages without kernel restart (try this first)
import sys

# Force reload the package path
if 'google.cloud.dataplex' in sys.modules:
    del sys.modules['google.cloud.dataplex']
if 'google.cloud.dataplex_v1' in sys.modules:
    del sys.modules['google.cloud.dataplex_v1']

# Now import
try:
    from google.cloud import dataplex_v1
    
    # Initialize Catalog client (for aspect types)
    catalog_client = dataplex_v1.CatalogServiceClient()
    
    print("✅ Dataplex Catalog client initialized")
    print(f"   Project: {PROJECT_ID}")
    print(f"   Region: {REGION}")
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print()
    print("Please restart the kernel and try again:")
    print("   - Click 'Kernel' → 'Restart' in the menu")
    print("   - Then run cells 1-3 again (config and auth)")
    print("   - Skip cell 15 (already installed)")
    print("   - Continue from this cell")
    raise

✅ Dataplex Catalog client initialized
   Project: johnswain-1200-20260303091357
   Region: us-central1


In [45]:
# Define the Aspect Type fields for Census Survey Metadata
print("🔧 Defining Census Survey Metadata Aspect Type...")
print()

# Construct the parent path (location)
parent = f"projects/{PROJECT_ID}/locations/{REGION}"
aspect_type_id = "census-survey-metadata"  # Note: only lowercase letters, numbers, and hyphens allowed

# Define each field as a MetadataTemplate
# Field 1: survey_vintage (int, required)
survey_vintage_field = dataplex_v1.AspectType.MetadataTemplate(
    name="survey_vintage",
    type="int",  # Valid types: datetime, double, bool, int, string, record, map, array, enum
    index=1,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        description="The year the data represents (e.g., 2022).",
        display_name="Survey Year (Vintage)"
    ),
    constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(
        required=True
    )
)

# Field 2: estimate_period (enum, required)
estimate_period_field = dataplex_v1.AspectType.MetadataTemplate(
    name="estimate_period",
    type="enum",
    index=2,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        description="The duration of the survey collection.",
        display_name="Estimate Period"
    ),
    constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(
        required=True
    ),
    enum_values=[
        dataplex_v1.AspectType.MetadataTemplate.EnumValue(
            index=1,
            name="1_year"
        ),
        dataplex_v1.AspectType.MetadataTemplate.EnumValue(
            index=2,
            name="5_year"
        )
    ]
)

# Field 3: is_experimental (bool)
is_experimental_field = dataplex_v1.AspectType.MetadataTemplate(
    name="is_experimental",
    type="bool",  # Valid types: datetime, double, bool, int, string, record, map, array, enum
    index=3,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        description="Set to true if the Census Bureau labels this as experimental (e.g., 2020 1-year).",
        display_name="Experimental Data"
    )
)

# Create the aspect type with all fields
aspect_type = dataplex_v1.AspectType(
    description="Captures the vintage and estimation type for US Census Bureau ACS datasets.",
    metadata_template=dataplex_v1.AspectType.MetadataTemplate(
        name="census_survey_metadata_template",
        type="record",
        record_fields=[survey_vintage_field, estimate_period_field, is_experimental_field]
    )
)

print(f"📋 Aspect Type Configuration:")
print(f"   ID: {aspect_type_id}")
print(f"   Description: {aspect_type.description}")
print(f"   Location: {parent}")
print(f"   Fields:")
print(f"     1. survey_vintage (int, required)")
print(f"     2. estimate_period (enum: 1_year, 5_year, required)")
print(f"     3. is_experimental (bool)")

🔧 Defining Census Survey Metadata Aspect Type...

📋 Aspect Type Configuration:
   ID: census-survey-metadata
   Description: Captures the vintage and estimation type for US Census Bureau ACS datasets.
   Location: projects/johnswain-1200-20260303091357/locations/us-central1
   Fields:
     1. survey_vintage (int, required)
     2. estimate_period (enum: 1_year, 5_year, required)
     3. is_experimental (bool)


In [46]:
# Create the Aspect Type
print()
print("🚀 Creating aspect type in Dataplex...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Create the aspect type using the catalog client
    create_operation = catalog_client.create_aspect_type(
        parent=parent,
        aspect_type=aspect_type,
        aspect_type_id=aspect_type_id
    )
    
    # Wait for the operation to complete
    print("   ⏳ Waiting for creation to complete...")
    result = create_operation.result(timeout=300)  # 5 minute timeout
    
    print()
    print("=" * 60)
    print("✅ Aspect Type created successfully!")
    print("=" * 60)
    print(f"   Resource name: {result.name}")
    print(f"   Description: {result.description}")
    print(f"   UID: {result.uid}")
    
except Exception as e:
    error_message = str(e)
    
    # Check if aspect type already exists
    if "ALREADY_EXISTS" in error_message or "already exists" in error_message:
        print()
        print("⚠️  Aspect Type already exists")
        print(f"   Name: projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{aspect_type_id}")
        print()
        print("   To update the aspect type, you'll need to delete it first:")
        print(f"   gcloud dataplex aspect-types delete {aspect_type_id} \\")
        print(f"     --project={PROJECT_ID} \\")
        print(f"     --location={REGION}")
        print()
        print("   Or continue - the existing aspect type will be used.")
    else:
        print(f"❌ Failed to create aspect type: {e}")
        raise


🚀 Creating aspect type in Dataplex...
   ⏳ Waiting for creation to complete...

✅ Aspect Type created successfully!
   Resource name: projects/johnswain-1200-20260303091357/locations/us-central1/aspectTypes/census-survey-metadata
   Description: Captures the vintage and estimation type for US Census Bureau ACS datasets.
   UID: 4851f2f6-b5d8-4e9f-85c3-01695132e432


In [47]:
# Validate the Aspect Type
print()
print("🔍 Validating Aspect Type...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Construct the full resource name
    aspect_type_name = f"projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{aspect_type_id}"
    
    # Get the aspect type to verify it exists
    retrieved_aspect_type = catalog_client.get_aspect_type(name=aspect_type_name)
    
    print()
    print("✅ Aspect Type verified:")
    print(f"   Resource name: {retrieved_aspect_type.name}")
    print(f"   Description: {retrieved_aspect_type.description}")
    print(f"   UID: {retrieved_aspect_type.uid}")
    print(f"   Created: {retrieved_aspect_type.create_time}")
    print()
    
    # Display the fields
    print("📋 Metadata Template Fields:")
    template = retrieved_aspect_type.metadata_template
    if template.record_fields:
        for field in sorted(template.record_fields, key=lambda f: f.index):
            required = " (required)" if field.constraints and field.constraints.required else ""
            print(f"   {field.index}. {field.name} ({field.type}){required}")
            if field.annotations:
                if field.annotations.display_name:
                    print(f"      Display: {field.annotations.display_name}")
                if field.annotations.description:
                    print(f"      Description: {field.annotations.description}")
            
            if field.enum_values:
                enum_names = [v.name for v in field.enum_values]
                print(f"      Allowed values: {', '.join(enum_names)}")
    
    print()
    print("✅ Checkpoint passed: Aspect Type created and validated!")
    
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise


🔍 Validating Aspect Type...

✅ Aspect Type verified:
   Resource name: projects/johnswain-1200-20260303091357/locations/us-central1/aspectTypes/census-survey-metadata
   Description: Captures the vintage and estimation type for US Census Bureau ACS datasets.
   UID: 4851f2f6-b5d8-4e9f-85c3-01695132e432
   Created: 2026-03-07 10:39:44.936258+00:00

📋 Metadata Template Fields:
   1. survey_vintage (int) (required)
      Display: Survey Year (Vintage)
      Description: The year the data represents (e.g., 2022).
   2. estimate_period (enum) (required)
      Display: Estimate Period
      Description: The duration of the survey collection.
      Allowed values: 1_year, 5_year
   3. is_experimental (bool)
      Display: Experimental Data
      Description: Set to true if the Census Bureau labels this as experimental (e.g., 2020 1-year).

✅ Checkpoint passed: Aspect Type created and validated!


## Create Public Data Governance Aspect Type

This aspect type captures governance metadata for external public datasets, including:

- **Source Agency**: The government agency or organization that published the data
- **License Type**: The data license (usually Public Domain for US Government data)
- **Last Cataloged**: When this dataset was last cataloged or verified

In [ ]:
# Define the Aspect Type fields for Public Data Governance
print("🔧 Defining Public Data Governance Aspect Type...")
print()

# Define the aspect type ID
governance_aspect_type_id = "data-governance-public"

# Field 1: source_agency (string)
source_agency_field = dataplex_v1.AspectType.MetadataTemplate(
    name="source_agency",
    type="string",
    index=1,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        display_name="Source Agency"
    )
)

# Field 2: data_license (string)
data_license_field = dataplex_v1.AspectType.MetadataTemplate(
    name="data_license",
    type="string",
    index=2,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        display_name="License Type",
        description="Usually Public Domain for US Government data."
    )
)

# Field 3: last_cataloged_date (datetime)
last_cataloged_field = dataplex_v1.AspectType.MetadataTemplate(
    name="last_cataloged_date",
    type="datetime",
    index=3,
    annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
        display_name="Last Cataloged"
    )
)

# Create the aspect type with all fields
governance_aspect_type = dataplex_v1.AspectType(
    description="Governance metadata for external public datasets.",
    metadata_template=dataplex_v1.AspectType.MetadataTemplate(
        name="data_governance_public_template",
        type="record",
        record_fields=[source_agency_field, data_license_field, last_cataloged_field]
    )
)

print(f"📋 Aspect Type Configuration:")
print(f"   ID: {governance_aspect_type_id}")
print(f"   Description: {governance_aspect_type.description}")
print(f"   Location: {parent}")
print(f"   Fields:")
print(f"     1. source_agency (string)")
print(f"     2. data_license (string)")
print(f"     3. last_cataloged_date (datetime)")

In [ ]:
# Create the Public Data Governance Aspect Type
print()
print("🚀 Creating Public Data Governance aspect type in Dataplex...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Create the aspect type using the catalog client
    create_operation = catalog_client.create_aspect_type(
        parent=parent,
        aspect_type=governance_aspect_type,
        aspect_type_id=governance_aspect_type_id
    )
    
    # Wait for the operation to complete
    print("   ⏳ Waiting for creation to complete...")
    result = create_operation.result(timeout=300)  # 5 minute timeout
    
    print()
    print("=" * 60)
    print("✅ Public Data Governance Aspect Type created successfully!")
    print("=" * 60)
    print(f"   Resource name: {result.name}")
    print(f"   Description: {result.description}")
    print(f"   UID: {result.uid}")
    
except Exception as e:
    error_message = str(e)
    
    # Check if aspect type already exists
    if "ALREADY_EXISTS" in error_message or "already exists" in error_message:
        print()
        print("⚠️  Aspect Type already exists")
        print(f"   Name: projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{governance_aspect_type_id}")
        print()
        print("   To update the aspect type, you'll need to delete it first:")
        print(f"   gcloud dataplex aspect-types delete {governance_aspect_type_id} \\")
        print(f"     --project={PROJECT_ID} \\")
        print(f"     --location={REGION}")
        print()
        print("   Or continue - the existing aspect type will be used.")
    else:
        print(f"❌ Failed to create aspect type: {e}")
        raise

In [ ]:
# Validate the Public Data Governance Aspect Type
print()
print("🔍 Validating Public Data Governance Aspect Type...")

# Check if catalog_client exists, if not initialize it
try:
    catalog_client
except NameError:
    print("   ℹ️  Initializing Dataplex Catalog client...")
    from google.cloud import dataplex_v1
    catalog_client = dataplex_v1.CatalogServiceClient()
    print("   ✅ Client initialized")

try:
    # Construct the full resource name
    governance_aspect_type_name = f"projects/{PROJECT_ID}/locations/{REGION}/aspectTypes/{governance_aspect_type_id}"
    
    # Get the aspect type to verify it exists
    retrieved_governance_aspect_type = catalog_client.get_aspect_type(name=governance_aspect_type_name)
    
    print()
    print("✅ Public Data Governance Aspect Type verified:")
    print(f"   Resource name: {retrieved_governance_aspect_type.name}")
    print(f"   Description: {retrieved_governance_aspect_type.description}")
    print(f"   UID: {retrieved_governance_aspect_type.uid}")
    print(f"   Created: {retrieved_governance_aspect_type.create_time}")
    print()
    
    # Display the fields
    print("📋 Metadata Template Fields:")
    template = retrieved_governance_aspect_type.metadata_template
    if template.record_fields:
        for field in sorted(template.record_fields, key=lambda f: f.index):
            required = " (required)" if field.constraints and field.constraints.required else ""
            print(f"   {field.index}. {field.name} ({field.type}){required}")
            if field.annotations:
                if field.annotations.display_name:
                    print(f"      Display: {field.annotations.display_name}")
                if field.annotations.description:
                    print(f"      Description: {field.annotations.description}")
    
    print()
    print("✅ Checkpoint passed: Public Data Governance Aspect Type created and validated!")
    
except Exception as e:
    print(f"❌ Validation failed: {e}")
    raise

In [ ]:
# Console links and next steps
print()
print("🔗 View in Google Cloud Console:")
print(f"   All Aspect Types: https://console.cloud.google.com/dataplex/govern/aspect-types?project={PROJECT_ID}")
print(f"   Census Survey Metadata: https://console.cloud.google.com/dataplex/govern/aspect-types/{aspect_type_id}?project={PROJECT_ID}")
print(f"   Public Data Governance: https://console.cloud.google.com/dataplex/govern/aspect-types/{governance_aspect_type_id}?project={PROJECT_ID}")
print()
print("✅ Section 3 complete! Both aspect types are ready:")
print("   1. Census Survey Metadata - for survey vintage and estimate period")
print("   2. Public Data Governance - for source agency and licensing info")
print()
print("📝 Next Steps:")
print("   - These aspect types can now be attached to BigQuery table entries")
print("   - Use the Dataplex Catalog API to apply aspects with metadata values")
print("   - Example: Attach survey_vintage=2018, estimate_period='5-Year Estimate' to blockgroup_2018_5yr")
print("   - Example: Attach source_agency='US Census Bureau', data_license='Public Domain' for governance")